# 📓 Semana 21 · Dia 1 — Motor de validações em 4 camadas

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | Portfólio empresarial |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Motor de validação rodando |

---


## 📖 Teoria — As 4 camadas do motor

| Camada | Valida | Exemplo |
|---|---|---|
| **1. Estrutura** | colunas/linhas existem | coluna obrigatória presente |
| **2. Tipo/formato** | valores convertem | data válida, número |
| **3. Consistência SQL** | coerência com o Lakehouse | produto existe, mês válido |
| **4. Regra de negócio** | regras Python do YAML | meta ≥ 0, desconto ≤ 100 |


### 💻 Na prática — Motor — camadas 1 e 2

Estrutura e tipo/formato.


In [ ]:
# Camada 1: estrutura
def valida_estrutura(dados, yaml_def):
    erros = []
    campos = {c["nome"]: c for c in yaml_def["campos"]}
    for campo, cfg in campos.items():
        if cfg.get("obrigatorio") and campo not in dados.columns:
            erros.append(f"Coluna obrigatória ausente: {campo}")
    return erros
print("Camada 1 pronta.")

In [ ]:
# Camada 2: tipo/formato
from datetime import datetime
def valida_tipo(valor, cfg):
    tipo = cfg["tipo"]
    if tipo == "numero":
        try:
            v = float(valor)
            if "min" in cfg and v < cfg["min"]:
                return f"menor que min {cfg['min']}"
            if "max" in cfg and v > cfg["max"]:
                return f"maior que max {cfg['max']}"
        except ValueError:
            return "não é número"
    if tipo == "data":
        try:
            datetime.strptime(str(valor), "%Y-%m-%d")
        except ValueError:
            return "data inválida (YYYY-MM-DD)"
    if tipo == "mes":
        if str(valor) not in [f"{m:02d}" for m in range(1, 13)]:
            return "mês inválido (01-12)"
    return None
print("Camada 2 pronta.")

### 💻 Na prática — Motor — camadas 3 e 4

Consistência com o Lakehouse e regras de negócio.


In [ ]:
# Camada 3: consistência SQL (produto existe?)
def valida_consistencia(produto):
    if produto is None: return None
    r = spark.sql(f"SELECT 1 FROM workspace.prata.dim_produto WHERE StockCode = '{produto}'").count()
    return None if r > 0 else f"produto {produto} não existe"
print("Camada 3 pronta (produto existe no catálogo).")

In [ ]:
# Camada 4: regras Python do YAML
def valida_regra_negocio(row, yaml_def):
    erros = []
    for campo, cfg in yaml_def["campos"].items():
        if cfg.get("obrigatorio") and (row.get(campo) in (None, "")):
            erros.append(f"{campo} obrigatório")
    return erros
print("Camada 4 pronta (estenda com regras custom do YAML).")

> 🎯 **Dica de prova**: Portfólio: explicar o motor em 4 camadas (estrutura → tipo → consistência → negócio) mostra design sólido — o entrevistador adora essa separação.


## 🎯 Exercícios de fixação

**1.** Adicione uma regra custom no YAML (ex.: desconto só para produto ativo).

**2.** Por que separar consistência SQL de regra Python?

**3.** Onde cada camada roda (Spark/engine)?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Regra custom

Adicione `regra: "produto_ativo"` no campo e trate na camada 4.

**2.** Separar

Consistência (SQL) é genérica e reutilizável; regra de negócio é específica do fluxo — separar facilita manutenção.

**3.** Onde roda

1–2: pandas/engine; 3: Spark SQL (Lakehouse); 4: Python.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*